In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
from itertools import product
from sklearn.base import BaseEstimator, RegressorMixin

In [2]:
# Not to use 'group. Instead use 'milkperiod','zdate','zdate_month'
df = pd.read_excel('E:\IUT\Lessons\Project-Bachelor\Husbandry\Dataset\TCI_sas (1).xlsx', sheet_name="Sheet1", usecols = ["milkperiod","zdate","zdate_month","firstmilk","firstmilkdays",
                                                                "prelendays","drylendays","milkdays",
                                                                "previous_Milk305",
                                                                "firstmilk_previous","SCS_305"])

In [3]:
df.isnull().sum()

milkperiod               0
zdate                    0
zdate_month              0
firstmilk                0
firstmilkdays            0
prelendays               0
drylendays            2059
milkdays              2059
previous_Milk305         0
firstmilk_previous       0
SCS_305                  0
dtype: int64

In [4]:
df['drylendays'].fillna(df['drylendays'].median(), inplace=True)
df['milkdays'].fillna(df['milkdays'].median(), inplace=True)

C:\Users\sepri\AppData\Local\Temp\ipykernel_21828\3106129121.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['drylendays'].fillna(df['drylendays'].median(), inplace=True)
C:\Users\sepri\AppData\Local\Temp\ipykernel_21828\3106129121.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as 

In [5]:
df.isnull().sum()

milkperiod            0
zdate                 0
zdate_month           0
firstmilk             0
firstmilkdays         0
prelendays            0
drylendays            0
milkdays              0
previous_Milk305      0
firstmilk_previous    0
SCS_305               0
dtype: int64

### **Split the data into train, validation, and test sets**

In [6]:
# Separate features and target
X = df.drop(columns=["firstmilk"])
y = df["firstmilk"].astype(float) 

# First split: separate test set (15%)
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42)

# Second split: separate train (70%) and validation (15%) from remaining data
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1765, random_state=42)  # 0.1765 ~ 15/(100-15)


### **Scale the features**

In [7]:
numerical_columns = ["milkperiod","zdate","zdate_month", "firstmilkdays", "prelendays", "drylendays", "milkdays", "previous_Milk305", "firstmilk_previous", "SCS_305"]

scaler = StandardScaler()

# Fit scaler on training data only
scaler.fit(X_train[numerical_columns])

# Transform train, validation, and test sets
X_train[numerical_columns] = scaler.transform(X_train[numerical_columns])
X_val[numerical_columns] = scaler.transform(X_val[numerical_columns])
X_test[numerical_columns] = scaler.transform(X_test[numerical_columns])

In [8]:
class RidgeRegressionGD(BaseEstimator, RegressorMixin):
    def __init__(self, learning_rate=0.01, epochs=1000, l2_lambda=0.1):
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.l2_lambda = l2_lambda
        self.weights = None
        self.bias = None

    def fit(self, X, y):
        n_samples, n_features = X.shape

        # Initialize weights and bias
        self.weights = np.zeros(n_features)
        self.bias = 0

        # Gradient Descent loop
        for i in range(self.epochs):
            y_pred = np.dot(X, self.weights) + self.bias
            error = y_pred - y

            # Compute gradients
            dw = (2 / n_samples) * (np.dot(X.T, error)) + 2 * self.l2_lambda * self.weights
            db = (2 / n_samples) * np.sum(error)

            if np.any(np.isnan(dw)) or np.any(np.isnan(db)):
                print("NaN detected in gradients")
                break

            # Update weights and bias
            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db

    def predict(self, X):
        return np.dot(X, self.weights) + self.bias
    
    def get_params(self, deep=True):
        return {
            'learning_rate': self.learning_rate,
            'epochs': self.epochs,
            'l2_lambda': self.l2_lambda
        }

    def set_params(self, **params):
        for param, value in params.items():
            setattr(self, param, value)
        return self


### **Define functions for additional metrics**

In [9]:
def calculate_mpe(y_true, y_pred):
    # Avoid division by zero
    mask = y_true != 0
    return np.mean((y_true[mask] - y_pred[mask]) / y_true[mask] * 100)

def calculate_smape(y_true, y_pred):
    # Avoid division by zero
    numerator = np.abs(y_true - y_pred)
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    mask = denominator != 0
    return np.mean(numerator[mask] / denominator[mask] * 100)

def calculate_sdr(y_true, y_pred):
    return np.std(y_pred) / np.std(y_true)

### **Train the initial model**

In [10]:
initial_model = RidgeRegressionGD(learning_rate=0.01, epochs=1000, l2_lambda=0.1)
initial_model.fit(X_train, y_train)

### **Prediction on test set**

In [11]:
y_test_pred = initial_model.predict(X_test)

### **Evaluate on test set**
test_mae = mean_absolute_error(y_test, y_test_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
test_r2 = r2_score(y_test, y_test_pred)
test_rmse = np.sqrt(test_mse)

# Calculate additional metrics for initial model
test_mpe = calculate_mpe(y_test, y_test_pred)
test_smape = calculate_smape(y_test, y_test_pred)
test_sdr = calculate_sdr(y_test, y_test_pred)

### **MPI evaluation (only Test Set)**
print("=" * 25)
print("Test Set (Ridge_Initial):")
print(f"R²    : {test_r2:.4f}")
print(f"MAE   : {test_mae:.4f}")
print(f"RMSE  : {test_rmse:.4f}")
print(f"MPE   : {test_mpe:.4f}")
print(f"sMAPE : {test_smape:.4f}")
print(f"SDR   : {test_sdr:.4f}")
print("=" * 25)

Test Set (Ridge_Initial):
R²    : 0.3110
MAE   : 7.0276
RMSE  : 9.2071
MPE   : -8.0856
sMAPE : 17.6840
SDR   : 0.5169


### **prediction on all datasets**

In [ ]:
y_train_pred = initial_model.predict(X_train)
y_val_pred = initial_model.predict(X_val)
y_test_pred = initial_model.predict(X_test)

### **Evaluate on all sets**

In [ ]:
train_mae = mean_absolute_error(y_train, y_train_pred)
train_mse = mean_squared_error(y_train, y_train_pred)
train_r2 = r2_score(y_train, y_train_pred)

val_mae = mean_absolute_error(y_val, y_val_pred)
val_mse = mean_squared_error(y_val, y_val_pred)
val_r2 = r2_score(y_val, y_val_pred)

test_mae = mean_absolute_error(y_test, y_test_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
test_r2 = r2_score(y_test, y_test_pred)

### **MPI evaluation**

In [ ]:
print("=" * 25)
print("Train Set:")
print(f"MAE  : {train_mae:.4f}")
print(f"MSE  : {train_mse:.4f}")
print(f"R²   : {train_r2:.4f}")
print("\nValidation Set:")
print(f"MAE  : {val_mae:.4f}")
print(f"MSE  : {val_mse:.4f}")
print(f"R²   : {val_r2:.4f}")
print("\nTest Set:")
print(f"MAE  : {test_mae:.4f}")
print(f"MSE  : {test_mse:.4f}")
print(f"R²   : {test_r2:.4f}")
print("=" * 25)

Train Set:
MAE  : 6.9779
MSE  : 83.4789
R²   : 0.3117

Validation Set:
MAE  : 6.9984
MSE  : 84.5055
R²   : 0.3058

Test Set:
MAE  : 7.0276
MSE  : 84.7686
R²   : 0.3110


### **Analyze overfitting**

In [ ]:
print("\nOverfitting Analysis:")
if val_mse > train_mse * 1.2 or test_mse > train_mse * 1.2:
    print("Warning: Potential overfitting detected! Validation or Test MSE is significantly higher than Train MSE.")
else:
    print("No significant overfitting detected. Train, Validation, and Test MSE are relatively close.")


Overfitting Analysis:
No significant overfitting detected. Train, Validation, and Test MSE are relatively close.


## **Grid Search for hyperparameter tuning**

In [ ]:
print("\nGrid Search for Hyperparameter Tuning:")
print("=" * 42)

param_grid = {
    'learning_rate': [0.001, 0.01, 0.1],
    'epochs': [500, 1000, 2000],
    'l2_lambda': [0.001, 0.01, 0.1]
}

# Initialize variables to store best parameters
best_params = None
best_val_mse = float('inf')
results = []

# Perform Grid Search
for lr, epochs, l2 in product(param_grid['learning_rate'], param_grid['epochs'], param_grid['l2_lambda']):
    # Train model with current hyperparameters
    model = RidgeRegressionGD(learning_rate=lr, epochs=epochs, l2_lambda=l2)
    model.fit(X_train, y_train)
    
    # Evaluate on validation set
    y_val_pred = model.predict(X_val)
    val_mse = mean_squared_error(y_val, y_val_pred)
    
    # Store results
    results.append({
        'learning_rate': lr,
        'epochs': epochs,
        'l2_lambda': l2,
        'val_mse': val_mse
    })
    
    # Update best parameters if current MSE is lower
    if val_mse < best_val_mse:
        best_val_mse = val_mse
        best_params = {'learning_rate': lr, 'epochs': epochs, 'l2_lambda': l2}

# Convert results to DataFrame
results_df = pd.DataFrame(results)

print("\nBest Hyperparameters:")
print(f"Learning Rate: {best_params['learning_rate']}")
print(f"Epochs: {best_params['epochs']}")
print(f"L2 Lambda: {best_params['l2_lambda']}")
print(f"Best Validation MSE: {best_val_mse:.4f}\n")
print("=" * 42)



Grid Search for Hyperparameter Tuning:

Best Hyperparameters:
Learning Rate: 0.1
Epochs: 2000
L2 Lambda: 0.001
Best Validation MSE: 84.2687



## **Train final model with best hyperparameters**

In [13]:
print("\nTraining Final Model with Best Hyperparameters:")
print("=" * 55)

best_params = {'learning_rate': 0.1, 'epochs': 2000, 'l2_lambda': 0.001}

final_model = RidgeRegressionGD(
    learning_rate=best_params['learning_rate'],
    epochs=best_params['epochs'],
    l2_lambda=best_params['l2_lambda']
)

final_model.fit(X_train, y_train)


Training Final Model with Best Hyperparameters:


### **Predict final model on test set**


In [14]:
y_test_pred_final = final_model.predict(X_test)

### **Final evaluation on test set**
test_mae_final = mean_absolute_error(y_test, y_test_pred_final)
test_mse_final = mean_squared_error(y_test, y_test_pred_final)
test_r2_final = r2_score(y_test, y_test_pred_final)
test_rmse_final = np.sqrt(test_mse_final)

# Calculate additional metrics for final model
test_mpe_final = calculate_mpe(y_test, y_test_pred_final)
test_smape_final = calculate_smape(y_test, y_test_pred_final)
test_sdr_final = calculate_sdr(y_test, y_test_pred_final)

### **Display final results (only Test Set)**
print("Final Model Evaluation (Test Set - Ridge_Optimized):")
print("=" * 60)
print(f"R²    : {test_r2_final:.4f}")
print(f"MAE   : {test_mae_final:.4f}")
print(f"RMSE  : {test_rmse_final:.4f}")
print(f"MPE   : {test_mpe_final:.4f}")
print(f"sMAPE : {test_smape_final:.4f}")
print(f"SDR   : {test_sdr_final:.4f}")
print("=" * 60)

Final Model Evaluation (Test Set - Ridge_Optimized):
R²    : 0.3134
MAE   : 7.0057
RMSE  : 9.1909
MPE   : -7.8779
sMAPE : 17.6642
SDR   : 0.5571


## **Print results into a new CSV file**


In [15]:
# Function to print existing CSV content
def print_csv_content(file_path):
    try:
        existing_df = pd.read_csv(file_path, index_col=0)
        return existing_df
    except FileNotFoundError:
        print("\nNo existing comparison table found.")
        return pd.DataFrame()

# Save results to a new CSV file
initial_results = {
    'Ridge_Initial': {
        'Test_R2': test_r2, 'Test_MAE': test_mae, 'Test_RMSE': test_rmse,
        'Test_MPE': test_mpe, 'Test_sMAPE': test_smape, 'Test_SDR': test_sdr
    }
}
initial_df = pd.DataFrame.from_dict(initial_results, orient='index')

final_results = {
    'Ridge_Optimized': {
        'Test_R2': test_r2_final, 'Test_MAE': test_mae_final, 'Test_RMSE': test_rmse_final,
        'Test_MPE': test_mpe_final, 'Test_sMAPE': test_smape_final, 'Test_SDR': test_sdr_final
    }
}
final_df = pd.DataFrame.from_dict(final_results, orient='index')

# Combine new results
new_results_df = pd.concat([initial_df, final_df])

# Save to a new file
new_file_path = 'test_evaluation_metrics.csv'
new_results_df.to_csv(new_file_path, index=True)
print("\n=== Updated Test Evaluation Metrics Saved to test_evaluation_metrics.csv ===")
print(new_results_df)


=== Updated Test Evaluation Metrics Saved to test_evaluation_metrics.csv ===
                  Test_R2  Test_MAE  Test_RMSE  Test_MPE  Test_sMAPE  Test_SDR
Ridge_Initial    0.310995  7.027556   9.206984 -8.085323   17.683976  0.516879
Ridge_Optimized  0.313395  7.005673   9.190935 -7.877927   17.664176  0.557122


### **Predict final model**

In [ ]:
y_train_pred_final = final_model.predict(X_train)
y_val_pred_final = final_model.predict(X_val)
y_test_pred_final = final_model.predict(X_test)

### **Final evaluation**

In [ ]:
train_mae_final = mean_absolute_error(y_train, y_train_pred_final)
train_mse_final = mean_squared_error(y_train, y_train_pred_final)
train_r2_final = r2_score(y_train, y_train_pred_final)

val_mae_final = mean_absolute_error(y_val, y_val_pred_final)
val_mse_final = mean_squared_error(y_val, y_val_pred_final)
val_r2_final = r2_score(y_val, y_val_pred_final)

test_mae_final = mean_absolute_error(y_test, y_test_pred_final)
test_mse_final = mean_squared_error(y_test, y_test_pred_final)
test_r2_final = r2_score(y_test, y_test_pred_final)

### **Display final results**

In [ ]:
print("Final Model Evaluation:")
print("=" * 60)
print("Train Set:")
print(f"MAE  : {train_mae_final:.4f}")
print(f"MSE  : {train_mse_final:.4f}")
print(f"R²   : {train_r2_final:.4f}")
print("\nValidation Set:")
print(f"MAE  : {val_mae_final:.4f}")
print(f"MSE  : {val_mse_final:.4f}")
print(f"R²   : {val_r2_final:.4f}")
print("\nTest Set:")
print(f"MAE  : {test_mae_final:.4f}")
print(f"MSE  : {test_mse_final:.4f}")
print(f"R²   : {test_r2_final:.4f}")
print("=" * 60)

print("\nFinal Model Parameters:")
print(f"Weights (coefficients): {final_model.weights}")
print(f"Bias (intercept)     : {final_model.bias:.4f}")

Final Model Evaluation:
Train Set:
MAE  : 6.9570
MSE  : 83.2274
R²   : 0.3137

Validation Set:
MAE  : 6.9766
MSE  : 84.2687
R²   : 0.3077

Test Set:
MAE  : 7.0057
MSE  : 84.4733
R²   : 0.3134

Final Model Parameters:
Weights (coefficients): [-1.8151773   0.19476734  0.33914441  3.2873116   2.06127142 -0.06625282
 -0.30893017  2.62905408  2.26441567 -0.49736479]
Bias (intercept)     : 42.7193


In [ ]:
def print_regression_formula(weights, bias, feature_names=None):
    n_features = len(weights)
    if feature_names is None:
        feature_names = [f"x{i+1}" for i in range(n_features)]

    terms = [f"{weights[i]:.4f}*{feature_names[i]}" for i in range(n_features)]
    formula = f"ŷ = {bias:.4f} + " + " + ".join(terms)
    print("\nFinal Regression Formula:")
    print(formula)

In [ ]:
print_regression_formula(final_model.weights, final_model.bias)


Final Regression Formula:
ŷ = 42.7193 + -1.8152*x1 + 0.1948*x2 + 0.3391*x3 + 3.2873*x4 + 2.0613*x5 + -0.0663*x6 + -0.3089*x7 + 2.6291*x8 + 2.2644*x9 + -0.4974*x10


## **Print results into csv fiel**

In [ ]:
# Function to print existing CSV content
def print_csv_content(file_path):
    existing_df = pd.read_csv(file_path, index_col=0)
    return existing_df

# Load existing CSV content from parent directory
file_path = '../model_comparison_table (1) (1).csv'
existing_df = print_csv_content(file_path)

# Save results to CSV (new results first)
# Add initial model results
initial_results = {
    'Ridge_Initial': {
        'Train_R2': train_r2, 'Train_MAE': train_mae, 'Train_MSE': train_mse,
        'Val_R2': val_r2, 'Val_MAE': val_mae, 'Val_MSE': val_mse,
        'Test_R2': test_r2, 'Test_MAE': test_mae, 'Test_MSE': test_mse
    }
}
initial_df = pd.DataFrame.from_dict(initial_results, orient='index')

# Add final model results
final_results = {
    'Ridge_Optimized': {
        'Train_R2': train_r2_final, 'Train_MAE': train_mae_final, 'Train_MSE': train_mse_final,
        'Val_R2': val_r2_final, 'Val_MAE': val_mae_final, 'Val_MSE': val_mse_final,
        'Test_R2': test_r2_final, 'Test_MAE': test_mae_final, 'Test_MSE': test_mse_final
    }
}
final_df = pd.DataFrame.from_dict(final_results, orient='index')

# Combine new results
new_results_df = pd.concat([initial_df, final_df])

# If existing data exists, append it; otherwise, start with new results
if not existing_df.empty:
    comparison_df = pd.concat([new_results_df, existing_df])
else:
    comparison_df = new_results_df

# Save to the parent directory file
comparison_df.to_csv(file_path, index=True)
print("\n=== Updated Model Comparison Table ===")
print(comparison_df)


=== Updated Model Comparison Table ===
                                                    Train_R2  Train_MAE  \
Ridge_Initial                                       0.311672   6.977889   
Ridge_Optimized                                     0.313746   6.956954   
RF_Initial                                          0.908400   2.525600   
RF_Tuned                                            0.379600   6.612000   
RF_CV_Final                                              NaN        NaN   
RF_Initial_with_grp                                 0.907400   2.541200   
RF_with_grp_Tuned                                   0.378200   6.620800   
RF_with_grp_CV_Final                                     NaN        NaN   
SVR_linear                                          0.307500   6.920500   
SVR_poly                                            0.267900   7.151600   
SVR_rbf                                             0.369600   6.559400   
XGBoost                                             0.416776